# Hands-on Lab: Generative AI for Data Repository Maintenance and Administration

## Scenario
Data repository management and administration is a crucial aspect of data engineering. It consists of all aspects that ensure the maintenance of the quality of the data, integrity of the data, and completeness of the data. You will explore the different aspects of data repository administration and management in the context of a real-world data set.

## Objectives
In this lab, you will learn how to use generative AI to generate codes that will do the following:
- Handle missing entries in the data
- Remove duplicate entries from the data
- Outlier detection in the data

## Testing Interface
The testing interface to be used for this lab is available in course page at the end of this lesson. Please keep that open in a browser on the side and follow the steps mentioned in it to set it up.

## Data set
The data set being used for this lab is a modified subset that maps the price of laptops with different attributes. The data set is a filtered and modified version of the “Laptop Price Prediction using specifications” data set, available under the Database Contents License (DbCL) v1.0 on the Kaggle website.

In [1]:
!pip install pandas requests

In [7]:
import io
import numpy as np
import pandas as pd
import requests
import subprocess

url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-AI0273EN-SkillsNetwork/labs/v1/m2/data/laptop_pricing_dataset_base.csv'

def get_content(url):
    try:
        resp = requests.get(url)
        resp.raise_for_status()
        return resp.text
    except Exception as e:
        print(f"Requests failed with error: {e}")
        print("Attempting fallback to system curl...")
        try:
            # Fallback to curl
            result = subprocess.run(['curl', '-L', url], capture_output=True, text=True, check=True)
            return result.stdout
        except subprocess.CalledProcessError as cpe:
            print(f"Curl also failed: {cpe}")
            raise e

def preprocess_data(df):
    # Identify columns with "?" values
    cols_with_question_mark = df.columns[df.isin(['?']).any()]
    print(cols_with_question_mark)
    # Replace "?" values with the mean value of the respective attribute
    for col in cols_with_question_mark:
        mean_val = pd.to_numeric(df[col], errors='coerce').mean()
        df[col] = pd.to_numeric(df[col].replace('?', mean_val))
    # Modify the data type of the attribute to float after replacement
    df[cols_with_question_mark] = df[cols_with_question_mark].astype(float)
    # Print the modified data
    print(df.dtypes)

def drop_duplicates(df):
    # Print the total number of rows before removal
    print("Total number of rows before removal:", len(df))
    # Remove duplicate entries
    df.drop_duplicates(inplace=True)
    # Print the total number of rows after removal
    print("Total number of rows after removal:", len(df))

# Write a python code to extract entries of a dataframe where the attribute 'Price' has outliers
# that might be anomalies in comparison to the other data.
def outlier_detection(df):
    mean_price = df['Price'].mean()
    std_price = df['Price'].std()
    outlier_threshold = 3

    outliers = df[(np.abs(df['Price'] - mean_price) > outlier_threshold * std_price)]
    entries_with_outliers = df[df['Price'].isin(outliers['Price'])]

    print("Outliers:")
    print(outliers)

    print("Entries with price outliers:")
    print(entries_with_outliers)

In [8]:
try:
    csv_content = get_content(url)
    df = pd.read_csv(io.StringIO(csv_content))
    print(df.head())

except Exception as e:
    print("Error fetching or reading CSV:", e)

  Manufacturer  Category     Screen  GPU  OS  CPU_core Screen_Size_cm  \
0         Acer         4  IPS Panel    2   1         5          35.56   
1         Dell         3    Full HD    1   1         3         39.624   
2         Dell         3    Full HD    1   1         7         39.624   
3         Dell         4  IPS Panel    2   1         5         33.782   
4           HP         4    Full HD    2   1         7         39.624   

   CPU_frequency  RAM_GB  Storage_GB_SSD Weight_kg  Price  
0            1.6       8             256       1.6    978  
1            2.0       4             256       2.2    634  
2            2.7       8             256       2.2    946  
3            1.6       8             128      1.22   1244  
4            1.8       8             256      1.91    837  


In [9]:

preprocess_data(df)

Index(['Screen_Size_cm', 'Weight_kg'], dtype='object')
Manufacturer       object
Category            int64
Screen             object
GPU                 int64
OS                  int64
CPU_core            int64
Screen_Size_cm    float64
CPU_frequency     float64
RAM_GB              int64
Storage_GB_SSD      int64
Weight_kg         float64
Price               int64
dtype: object


In [10]:
drop_duplicates(df)

Total number of rows before removal: 243
Total number of rows after removal: 238


In [11]:
outlier_detection(df)

Outliers:
    Manufacturer  Category     Screen  GPU  OS  CPU_core  Screen_Size_cm  \
64          Asus         1    Full HD    3   1         7          43.942   
77          Dell         5    Full HD    3   1         7          43.942   
144       Lenovo         3  IPS Panel    3   1         7          43.180   
159        Razer         1    Full HD    3   1         7          35.560   

     CPU_frequency  RAM_GB  Storage_GB_SSD  Weight_kg  Price  
64             2.9      16             256       3.60   3810  
77             2.9      16             256       3.42   3665  
144            2.8       8             256       3.40   3810  
159            2.8      16             256       1.95   3301  
Entries with price outliers:
    Manufacturer  Category     Screen  GPU  OS  CPU_core  Screen_Size_cm  \
64          Asus         1    Full HD    3   1         7          43.942   
77          Dell         5    Full HD    3   1         7          43.942   
144       Lenovo         3  IPS Panel